# Feature Engineering — Lead-Slot Ranker (Milestone 1, Section 3)

Builds the feature table for the `is_lead` ranking task (does an article win the lead
position of its Google News story cluster?) from the raw `Behind-the-Feed-Data-Challenge`
scrape. Implements the role-separated plan from the written feature-engineering doc:

- **Merit features** — fed to the ranking score (timing, headline content, story context)
- **Fairness-constraint features** — audited and used to calibrate exposure, **never** fed
  to the score (outlet identity, ownership, payment history, the outlet's own trailing lead
  rate) — feeding these to the score would let a model relearn today's outlet-tier bias
  instead of correcting it
- **Evaluation-only features** — track the Online News Act trend, not score inputs

Windows note: install
`sentence-transformers` into a virtual environment at a **short path** (e.g. `C:\venv`); If not installed the fallback covers this
automatically if it happens, but the real model is what actually produced the committed
`outputs/engineered_features_is_lead.csv`.

**Run this notebook from the repo root** (same convention as `main_findings.ipynb`):

In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

DATA = 'data'  # relative to the repo root, same convention as main_findings.ipynb
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)

FRONT_PAGE = Path(DATA) / 'google_news_front_page'
assert FRONT_PAGE.exists(), (
    f"Could not find {FRONT_PAGE} — run this notebook from the repo root "
    "(the folder that contains data/ and outputs/)."
)

## 1. Rebuild the story-cluster base table

Same cluster definition `main_findings.ipynb` uses for the lead-ratio analysis, so the
features below describe exactly the population that metric is computed over: Top
Stories cards from `purpose == 'main'`, non-excluded snapshots, with YouTube videos
pulled out of the competing set (a video in the lead slot isn't an "article" competing
for a text-story lead).

In [2]:
art = pd.read_csv(FRONT_PAGE / 'front_page_articles.csv', dtype={'timestamp': str})
snaps = pd.read_csv(FRONT_PAGE / 'snapshots.csv', dtype={'timestamp': str})
outlets = pd.read_csv(FRONT_PAGE / 'outlets.csv')

use = snaps[(snaps.purpose == 'main') & snaps.downloaded.astype(str).eq('True') & ~snaps.exclude.astype(str).eq('True')]
s = art[art.timestamp.isin(use.timestamp) & art.section.isin(['Top stories', 'À la une'])].copy()
s['group'] = s.group.fillna('unknown')
s['video'] = s.outlet.astype(str).str.startswith('youtube.com')

key = ['timestamp', 'card']
n_nonvideo = s[~s.video].groupby(key).size().rename('n_nonvideo')
video_led = s[s.video & (s.slot == 0)].groupby(key).size().rename('video_led')
s = s.join(n_nonvideo, on=key).join(video_led, on=key)
s['cluster_size_nonvideo'] = s.n_nonvideo.fillna(0).astype(int)
s['in_cluster'] = ~s.video & (s.cluster_size_nonvideo >= 2) & s.video_led.isna()

cl = s[s.in_cluster].copy()
cl['cluster_id'] = cl.timestamp + '_' + cl.card.astype(str)
print(f'{cl.cluster_id.nunique():,} story clusters, {len(cl):,} article rows')

2,175 story clusters, 8,426 article rows


## 2. Target

`is_lead` = did this article occupy slot 0 (the lead) of its story cluster?

In [3]:
cl['is_lead'] = (cl.slot == 0).astype(int)
print(cl.is_lead.value_counts(normalize=True).rename('share'))

is_lead
0    0.74187
1    0.25813
Name: share, dtype: float64


## 3. Merit features — fed to the ranking score

**Timing — investigate first.** How long after a cluster's first-observed publish time
does a typical competing article appear? Before picking any cap, look at the raw gap.

In [4]:
raw_gap_min = (cl.published_unix - cl.groupby('cluster_id').published_unix.transform('min')) / 60.0
print(raw_gap_min.describe())
print('99th percentile (minutes):', round(raw_gap_min.quantile(0.99), 1))
worst = raw_gap_min.sort_values(ascending=False).index[:3]
cl.loc[worst, ['headline', 'timestamp']]

count    8.426000e+03
mean     1.838161e+03
std      6.146639e+04
min      0.000000e+00
25%      0.000000e+00
50%      2.698583e+02
75%      8.841292e+02
max      3.954877e+06
Name: published_unix, dtype: float64
99th percentile (minutes): 7515.8


,headline,timestamp
7054,Jury selection begins in Trump's 1st criminal ...,20240415181715
7053,Donald Trump calls hush money case an 'assault...,20240415181715
8994,Dozens killed in Russian strike on military ed...,20240903202821


<!-- The median gap is minutes, the 99th percentile is still under a day — but the raw
max is in the millions of minutes (years), and the worst rows above are dated years
before the 2022–2026 snapshot window: stale publisher metadata on a reposted/evergreen
article, not a real same-story gap.  -->
**Decision:** cap at 14 days (generous against the
~2-day snapshot cadence, so it never touches a genuine gap) and keep a flag, so capped
rows stay identifiable rather than silently indistinguishable from a real same-day gap.

In [5]:
t = pd.to_numeric(cl.published_unix, errors='coerce')
cl['published_unix'] = t
cl['cluster_min_t'] = cl.groupby('cluster_id').published_unix.transform('min')
cl['is_first_published'] = (cl.published_unix == cl.cluster_min_t).astype(int)

gap_min = (cl.published_unix - cl.cluster_min_t) / 60.0
MAX_PLAUSIBLE_GAP_MIN = 14 * 24 * 60  # 14 days
cl['stale_timestamp_flag'] = (gap_min > MAX_PLAUSIBLE_GAP_MIN).astype(int)
cl['time_since_first_publish_min'] = gap_min.clip(upper=MAX_PLAUSIBLE_GAP_MIN)  # TRANSFORM: capped/clipped (outlier containment)

print(f"{cl.stale_timestamp_flag.sum()} rows capped (stale/erroneous publish timestamp)")
cl[['time_since_first_publish_min', 'is_first_published']].describe()

17 rows capped (stale/erroneous publish timestamp)


,time_since_first_publish_min,is_first_published
count,8426.000000,8426.000000
mean,730.606460,0.261453
std,1563.895426,0.439452
min,0.000000,0.000000
25%,0.000000,0.000000
50%,269.858333,0.000000
75%,884.129167,1.000000
max,20160.000000,1.000000


**Headline content — investigate first.** Is this actually a bilingual corpus, or is
French rare enough that a plain English vectorizer would do?

In [6]:
print(cl.edition.value_counts())
print((cl.edition == 'fr-CA').mean().round(3), 'share French')

edition
en-CA    4917
fr-CA    3509
Name: count, dtype: int64
0.416 share French


About 42% of rows are French — not a rare edge case, so a language-agnostic
or genuinely multilingual representation earns its place here, not just a nice-to-have.


**Decision:** embed with a real multilingual sentence embedding, 384-dimension, from
`paraphrase-multilingual-MiniLM-L12-v2` (the multilingual-MiniLM class named in the
written plan), so French and English headlines about the same event land close together
in vector space without a separate per-language pipeline or translation step. 
<!-- If
`sentence-transformers` isn't installed or fails to load (see the Windows path-length
note above), this falls back to a dependency-free hashing-trick character-trigram
vectorizer at the same 384-dim width, so `headline_emb_*` always has the same shape
regardless of which path ran — the notebook prints which one it used. -->

In [7]:
EMB_DIM = 384  # matches paraphrase-multilingual-MiniLM-L12-v2's output width

def hash_embed(text, dim=EMB_DIM, n=3):
    """Dependency-free fallback: bag-of-character-trigrams, hashed into a fixed-width vector."""
    text = str(text).lower()
    vec = np.zeros(dim)
    for i in range(len(text) - n + 1):
        gram = text[i:i + n]
        h = int(hashlib.md5(gram.encode('utf-8')).hexdigest(), 16)  # md5, not Python's hash(): stable across runs
        vec[h % dim] += 1.0
    norm = np.linalg.norm(vec)
    return vec / norm if norm > 0 else vec  # TRANSFORM: L2-normalized (fallback embedding only)

cl['headline_length'] = cl.headline.astype(str).str.len()
headlines = cl.headline.astype(str).tolist()

try:
    from sentence_transformers import SentenceTransformer
    _model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    emb = _model.encode(headlines, batch_size=64, show_progress_bar=False)  # TRANSFORM: text -> 384-dim embedding
    embedding_method = 'sentence-transformers (paraphrase-multilingual-MiniLM-L12-v2)'
except Exception as exc:
    print(f'sentence-transformers unavailable ({exc!r}) — falling back to the hashing-trick vectorizer')
    emb = np.vstack([hash_embed(h) for h in headlines])  # TRANSFORM: text -> 384-dim embedding (fallback)
    embedding_method = 'hashing-trick fallback (sentence-transformers unavailable)'

print(f'headline embedding method: {embedding_method}')
emb_cols = [f'headline_emb_{i}' for i in range(EMB_DIM)]
# one concat for all 384 columns at once, instead of looping column-by-column
# (the latter fragments the DataFrame badly enough at this width to warn on every later assignment)
cl = pd.concat([cl, pd.DataFrame(emb, columns=emb_cols, index=cl.index)], axis=1)
print(f'headline embedding shape: {emb.shape}')

c:\embtest\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3734.14it/s]


headline embedding method: sentence-transformers (paraphrase-multilingual-MiniLM-L12-v2)
headline embedding shape: (8426, 384)


**Story context.** `cluster_size_nonvideo` (how many outlets are competing for this
story — a lead in a 2-way cluster and a lead in a 6-way cluster aren't equally hard;
named distinctly from the raw CSV's own `cluster_size` column, which counts videos too),
`section`, `edition` and `card` (the story's own position on the page, not to be
confused with `slot`, which defines the target). `section`/`edition` are one-hot
encoded here as the plan's categorical-encoding example — both are small, non-ordinal
sets.

In [8]:
cat_merit = pd.get_dummies(cl[['section', 'edition']], prefix=['section', 'edition'])  # TRANSFORM: one-hot encoded
cl = pd.concat([cl, cat_merit], axis=1)
print(cat_merit.columns.tolist())

['section_Top stories', 'section_À la une', 'edition_en-CA', 'edition_fr-CA']


## 4. Fairness-constraint features — audited, never fed to the score

**Outlet identity — investigate first.** Before deciding `group` must be held out of the
score, check whether it actually predicts the outcome on its own — if it didn't, there'd
be nothing to hold out.

In [9]:
cl.groupby('group').is_lead.agg(['mean', 'count']).sort_values('mean', ascending=False)

,mean,count
group,,
major,0.288792,5710
foreign,0.244831,1838
cdn_unpaid,0.100529,189
franco,0.091858,479
local_en,0.062992,127
unknown,0.060241,83


`major` leads at roughly double the rate of the smaller/local groups — on `group`
alone, with nothing about the specific story yet. We need to prevent the model from exploiting this pattern: if `group` were handed to it as an ordinary input: it would learn
"major → lead" as a shortcut instead of learning from timing or content, and a ranker
built on it would reproduce today's bias rather than correct it. 

**Decision:** `group`/
`group_alt` (payment tier), the outlet's own historical slot frequency and CJC payment
amount, how concentrated ownership is among the competitors in *this* cluster (Herfindahl
index over `payee_legal` shares), how often this outlet co-occurs with others in the same
story clusters (a simple weighted-degree centrality, no graph library needed), and the
outlet's own trailing lead rate computed **causally** all stay out of the score and are
used only to audit and calibrate it afterward.

In [10]:
# outlet's own historical slot frequency (cjc_y2 / payee_legal already ride along on each
# article row from the collection step — only n_slots needs merging in from outlets.csv)
cl = cl.merge(outlets[['domain', 'n_slots']].rename(columns={'domain': 'outlet'}), on='outlet', how='left')
print(cl[['n_slots', 'cjc_y2']].describe().loc[['mean', '50%', 'max']])  # mean >> median on both: heavy right skew

          n_slots        cjc_y2
mean   783.002018  3.641783e+06
50%    690.000000  2.728434e+06
max   1670.000000  1.029535e+07


Both columns have a mean above the median, and a max several times either —
a handful of large national outlets stretch the top of both scales. **Decision:** `log1p`,
so that stretch compresses instead of dominating a linear scale.

In [11]:
cl['n_slots_log'] = np.log1p(cl.n_slots.fillna(0))  # TRANSFORM: log1p (right-skew compression)
cl['cjc_payment_log'] = np.log1p(cl.cjc_y2.fillna(0).clip(lower=0))  # TRANSFORM: log1p (right-skew compression)

# ownership concentration among the competitors in each specific cluster
def hhi(owners):
    shares = owners.value_counts(normalize=True)
    return float((shares ** 2).sum())

cl = cl.join(cl.groupby('cluster_id').payee_legal.apply(hhi).rename('owner_hhi_in_cluster'), on='cluster_id')

# outlet co-occurrence weight: how often does this outlet share a cluster with others
# (a self-join on cluster_id, excluding the row's own outlet — a plain-pandas stand-in
# for graph-style centrality, no networkx dependency needed)
pairs = cl.merge(cl[['cluster_id', 'outlet']], on='cluster_id', suffixes=('', '_b'))
pairs = pairs[pairs.outlet != pairs.outlet_b]
weight = pairs.groupby('outlet').size().rename('cooccurrence_weight')
cl = cl.join(weight, on='outlet')
cl['outlet_cooccurrence_centrality'] = cl.cooccurrence_weight.fillna(0) / cl.cooccurrence_weight.max()  # TRANSFORM: min-max normalized to [0, 1]

**Trailing lead rate — investigate first.** This one has to be computed causally
(only using rows *before* the current one in time) or it leaks the very outcome it's
meant to audit. To see why concretely, compare it against the naive version a first
draft would be tempted to write: one `groupby('outlet').is_lead.transform('mean')` over
*all* rows, future ones included.

In [12]:
cl_time_sorted = cl.sort_values('published_unix')
naive_rate = cl_time_sorted.groupby('outlet').is_lead.transform('mean')  # leaky: uses the whole outlet history, including this row and the future
causal_rate = (
    cl_time_sorted.groupby('outlet').is_lead.cumsum() - cl_time_sorted.is_lead
) / cl_time_sorted.groupby('outlet').cumcount().replace(0, np.nan)
diff = (naive_rate - causal_rate.fillna(0)).abs()
print('mean |naive - causal| gap:', round(diff.mean(), 3), '| rows where naive is well above causal:', int((naive_rate - causal_rate.fillna(0) > 0.2).sum()))

mean |naive - causal| gap: 0.045 | rows where naive is well above causal: 185


The naive version disagrees with the causal one by a non-trivial margin for plenty of
rows — exactly the rows where today's lead rate was still being decided partly by leads
the model wouldn't actually have seen yet at the time. 

**Decision:** keep only the causal
version below; `cumsum`/`cumcount` both respect the outlet grouping, and subtracting the
row's own label from the cumulative sum removes it from its own rate without a per-group
`shift()` (which is easy to get wrong across group boundaries — an earlier draft of this
notebook did exactly that).

In [13]:
cl = cl.sort_values('published_unix')
cl['outlet_prior_leads'] = cl.groupby('outlet').is_lead.cumsum() - cl.is_lead
cl['outlet_prior_count'] = cl.groupby('outlet').cumcount()
cl['trailing_lead_rate'] = (cl.outlet_prior_leads / cl.outlet_prior_count).replace([np.inf, -np.inf], np.nan).fillna(0)  # TRANSFORM: causal (backward-looking only) rate

cl[['n_slots_log', 'cjc_payment_log', 'owner_hhi_in_cluster', 'outlet_cooccurrence_centrality', 'trailing_lead_rate']].describe()

,n_slots_log,cjc_payment_log,owner_hhi_in_cluster,outlet_cooccurrence_centrality,trailing_lead_rate
count,8426.000000,8426.000000,8426.000000,8426.000000,8426.000000
mean,5.903667,11.366279,0.417194,0.459548,0.258301
std,1.717867,6.612694,0.256762,0.334386,0.157656
min,0.693147,0.000000,0.000000,0.000462,0.000000
25%,4.955827,0.000000,0.250000,0.120037,0.150134
50%,6.538140,14.819239,0.333333,0.460295,0.285714
75%,7.268920,15.765119,0.500000,0.825485,0.352035
max,7.421178,16.147203,1.000000,1.000000,1.000000


## 5. Evaluation-only features — track the trend, not score inputs

Which Online News Act era a snapshot falls in, in two forms: a **continuous**
days-since-each-milestone transform (per the written plan's transformations table) and
the **categorical** period bucket `main_findings.ipynb` already reports its Figure 4
trend by. Neither is a ranking input — the policy era an article happened to appear in
says nothing about that article's own merit; this is for reporting the trend only.

In [14]:
MILESTONES = {
    'meta_block': ('2023-08-01', 'Meta block'),
    'google_agreement': ('2023-11-29', 'Google agreement'),
    'cjc_designated': ('2024-06-07', 'CJC designated'),
    'payments_flowing': ('2025-03-11', 'Payments flowing'),
}
day = pd.to_datetime(cl.timestamp.str[:8])

# continuous: days since each milestone (negative = before it happened yet)
days_since_cols = []
for key, (date, _) in MILESTONES.items():
    col = f'days_since_{key}'
    cl[col] = (day - pd.Timestamp(date)).dt.days  # TRANSFORM: absolute date -> milestone-relative day offset
    days_since_cols.append(col)

# categorical: which era the snapshot falls in (same bins as main_findings.ipynb Fig. 4)
PERIODS = [('2022-01-01', 'Before Meta block')] + [(date, label) for _, (date, label) in MILESTONES.items()]
cl['period'] = pd.cut(  # TRANSFORM: date -> 5-bucket ordinal/categorical period
    day,
    bins=[pd.Timestamp(p) for p, _ in PERIODS] + [pd.Timestamp('2030-01-01')],
    labels=[name for _, name in PERIODS],
    right=False,
)
print(cl[days_since_cols].describe().loc[['min', 'max']])
cl.period.value_counts().reindex([name for _, name in PERIODS])

     days_since_meta_block  days_since_google_agreement  \
min                 -442.0                       -562.0   
max                 1165.0                       1045.0   

     days_since_cjc_designated  days_since_payments_flowing  
min                     -753.0                      -1030.0  
max                      854.0                        577.0  


period
Before Meta block    1256
Meta block            480
Google agreement     1250
CJC designated       1798
Payments flowing     3642
Name: count, dtype: int64

## 6. Assemble the role-separated feature tables

Kept as **separate DataFrames**, not just separate columns in one table, so a later
training step can't accidentally concatenate a protected attribute into the score
model's inputs by forgetting to drop a column — `X_merit` is the only one that should
ever reach a ranking model.

In [15]:
ID_COLS = ['cluster_id', 'timestamp', 'outlet', 'article_id', 'story_id']

merit_cols = (
    ['cluster_size_nonvideo', 'card', 'time_since_first_publish_min', 'is_first_published',
     'stale_timestamp_flag', 'headline_length'] + emb_cols + list(cat_merit.columns)
)
constraint_cols = [
    'group', 'group_alt', 'n_slots_log', 'cjc_payment_log',
    'owner_hhi_in_cluster', 'outlet_cooccurrence_centrality', 'trailing_lead_rate',
]
eval_cols = ['period'] + days_since_cols

X_merit = cl[ID_COLS + merit_cols].copy()
audit_constraints = cl[ID_COLS + constraint_cols].copy()
eval_context = cl[ID_COLS + eval_cols].copy()
y = cl[ID_COLS + ['is_lead']].copy()

print('X_merit', X_merit.shape, '| audit_constraints', audit_constraints.shape, '| eval_context', eval_context.shape)
X_merit.head(3)

X_merit (8426, 399) | audit_constraints (8426, 12) | eval_context (8426, 10)


,cluster_id,timestamp,outlet,article_id,story_id,cluster_size_nonvideo,card,time_since_first_publish_min,is_first_published,stale_timestamp_flag,...,headline_emb_378,headline_emb_379,headline_emb_380,headline_emb_381,headline_emb_382,headline_emb_383,section_Top stories,section_À la une,edition_en-CA,edition_fr-CA
2619,20240415181715_0,20240415181715,washingtonpost.com,CBMiqAFodHRwczovL3d3dy53YXNoaW5ndG9ucG9zdC5jb2...,CAAqNggKIjBDQklTSGpvSmMzUnZjbmt0TXpZd1NoRUtEd2...,3,0,0.0,1,0,...,0.208700,0.491702,0.086817,-0.062824,-0.098588,0.221599,True,False,True,False
5,20220516062725_1,20220516062725,lapresse.ca,CBMidmh0dHBzOi8vd3d3LmxhcHJlc3NlLmNhL2ludGVybm...,CAAqNggKIjBDQklTSGpvSmMzUnZjbmt0TXpZd1NoRUtEd2...,3,1,0.0,1,0,...,-0.392868,0.024702,0.209937,0.000102,0.137552,0.273643,False,True,False,True
1,20220516062725_0,20220516062725,lapresse.ca,CBMigQFodHRwczovL3d3dy5sYXByZXNzZS5jYS9pbnRlcm...,CAAqNggKIjBDQklTSGpvSmMzUnZjbmt0TXpZd1NoRUtEd2...,3,0,0.0,1,0,...,0.285735,0.053878,-0.131707,-0.111116,-0.101361,-0.103474,False,True,False,True


## 7. Data-quality check

A quick missingness pass on the engineered columns before saving — the milestone's
data-issues plan calls for checking this rather than assuming a clean merge.

In [16]:
missing = pd.concat([X_merit[merit_cols], audit_constraints[constraint_cols]], axis=1).isna().mean().sort_values(ascending=False)
missing = missing[missing > 0]
print('Columns with missing values (share of rows):' if len(missing) else 'No missing values in any engineered column.')
missing

No missing values in any engineered column.


Series([], dtype: float64)

## 8. Save the output artifacts

Two files, so the headline-embedding width (384 real dims, or 384 hashed dims on the
fallback path) doesn't bloat the main table into something unreadable in a diff or a
quick `pd.read_csv` peek:

- **`engineered_features_is_lead.csv`** — everything except the embedding, one combined
  table with each column's role kept explicit in its name (`merit__*`, `constraint__*`,
  `eval__*`) so the separation survives being written to a single CSV. A model-training
  notebook should still only ever select the `merit__*` columns as `X`.
- **`headline_embeddings_is_lead.npz`** — the embedding matrix plus `article_id` so it can
  be joined back to the CSV by that key (safer than assuming row order survives a reload).

In [17]:
non_embedding_merit_cols = [c for c in merit_cols if not c.startswith('headline_emb_')]

combined = pd.concat([
    y,
    X_merit[non_embedding_merit_cols].add_prefix('merit__'),
    audit_constraints[constraint_cols].add_prefix('constraint__'),
    eval_context[eval_cols].add_prefix('eval__'),
], axis=1)

csv_path = OUT / 'engineered_features_is_lead.csv'
combined.to_csv(csv_path, index=False)
print(f'Saved {len(combined):,} rows x {combined.shape[1]} columns to {csv_path}')

emb_path = OUT / 'headline_embeddings_is_lead.npz'
np.savez_compressed(emb_path, article_id=cl.article_id.to_numpy(), embedding=emb, method=embedding_method)
print(f'Saved {emb.shape} embedding matrix ({embedding_method}) to {emb_path}')

Saved 8,426 rows x 28 columns to outputs\engineered_features_is_lead.csv
Saved (8426, 384) embedding matrix (sentence-transformers (paraphrase-multilingual-MiniLM-L12-v2)) to outputs\headline_embeddings_is_lead.npz


## 9. Transformation log

Every `# TRANSFORM:`-tagged cell above, in one place:

| Column(s) | Transformation | Why (see the "investigate first" cell above it) |
| --- | --- | --- |
| `time_since_first_publish_min` | Capped at 14 days, flagged | A few publish timestamps were years off (stale metadata); the cap stops them from dominating the scale |
| `headline_emb_0..383` | Text → 384-dim embedding (model, or hashing-trick fallback) | ~1 in 3 headlines are French — a language-agnostic/multilingual vector earns its place |
| (fallback path only) `headline_emb_0..383` | L2-normalized | Keeps the hashed bag-of-trigrams vector on a comparable scale to the real model's output |
| `section`, `edition` | One-hot encoded | Small, non-ordinal categorical sets |
| `n_slots_log`, `cjc_payment_log` | `log1p` | Mean far above median on both — a handful of large outlets dominate the raw scale |
| `outlet_cooccurrence_centrality` | Min-max normalized to [0, 1] | Comparable scale across outlets regardless of raw cluster-appearance counts |
| `trailing_lead_rate` | Causal (backward-looking only) cumulative rate | A naive whole-history version leaks the future into a row's own feature — shown concretely above |
| `days_since_<milestone>` | Absolute date → signed day offset per milestone | The Online News Act's effect is relative to its own milestones, not the calendar date |
| `period` | Date → 5-bucket categorical | Matches the era grouping `main_findings.ipynb` already reports its trend by |